In [32]:
import ast
import json
from glob import glob

import pandas as pd
import os

In [95]:
BASE = "/Users/charles-/Downloads/Master/M2/PersonasData/Mac Miller"
CURRENT_ARTIST = "MacMiller"
SONGS_EXT = "tracks/songsWithRhymeSchemes/*.json"

In [ ]:
PERSONAS = ["Mac_Miller", "Delusional_Thomas", "Larry_Lovestein"] # identifiants de persona qu'on veut donner à chacune

## Création persona_attribution_persona.csv

In [ ]:
writer_artist_ids = ["[820]", "[820]", "[4389143]"] # à chercher avec un tracksDfs[i] > writer_artists_id. 

In [ ]:
# Station de nettoyage des featurings. Non obligatoire si vous êtes moins des sauvages que moi et que vous avez trié à la main
tracksDfs = [
    df[df["writer_artists_id"] == writer_id]
    for df, writer_id in zip(tracksDfs, writer_artist_ids)
]

In [ ]:
personaFolders = [os.path.join(BASE, persona) for persona in PERSONAS]
tracksDfsPaths = [os.path.join(persona_folder, "tracksDf.csv") for persona_folder in personaFolders]
tracksDfs = [pd.read_csv(path) for path in tracksDfsPaths]

In [ ]:
persona_attribution_dfs = [
    pd.DataFrame({
        "id": df["id"],
        "title": df["title"],
        "album_id": df["album_id"],
        "persona_id": persona,  # scalar is broadcast, no need for [persona]*l
    })
    for persona, df in zip(PERSONAS, tracksDfs)
]

In [ ]:
persona_attribution_df = pd.concat(persona_attribution_dfs)

In [ ]:
persona_attribution_df.to_csv(f"{CURRENT_ARTIST}_attribution_persona.csv")

# Ajout persona dans les fichiers de chansons

In [ ]:
def add_persona_id(sections, persona_id):
    for section in sections:
        for line in section.get("lines", []):
            line["persona"] = persona_id
    return sections


def load_and_apply_persona(path_pattern, ids, persona_id):
    """Charge les JSON, garde les ids voulus et ajoute la persona à chaque ligne."""
    rows = []
    for path in glob(path_pattern):
        with open(path, encoding="utf-8") as f:
            song = json.load(f)
        if song["song_id"] in ids:
            song["lyrics"] = add_persona_id(song["lyrics"], persona_id)
            rows.append(song)
    return pd.DataFrame(rows)

In [71]:
attribution_df = pd.read_csv(f"{CURRENT_ARTIST}_attribution_persona.csv")

In [101]:
dfs = list()
for persona in PERSONAS:
    path_to_persona = os.path.join(BASE, persona)
    path_to_songs = os.path.join(path_to_persona, SONGS_EXT)
    songs_ids_persona = attribution_df[attribution_df["persona_id"] == persona]["id"].to_list()
    df_persona = load_and_apply_persona(path_to_songs, songs_ids_persona, persona_id=persona)
    dfs.append(df_persona)

artist_df = pd.concat(dfs)

In [ ]:
# Sauvegarde : lyrics sérialisé en vrai JSON
artist_df.assign(lyrics=artist_df["lyrics"].apply(json.dumps, ensure_ascii=False)).to_csv(f"{BASE}/artist_df.csv", index=False)

# Passage du dataframe artiste à des json par chanson pour mesures

In [ ]:
for sid, group in artist_df.groupby("song_id"):
    group.iloc[0].to_json(f"Corpus/{CURRENT_ARTIST}_songs/{sid}.json", force_ascii=False, indent=2)